# Practice 109 — Bayesian Causal Inference & Structural Time Series

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import numpy as np
import arviz as az
import xy.pyplot as plt

from src.datasets import load_dataset
from src.plotting import (
    observed_vs_counterfactual_plot,
    cumulative_effect_plot,
    effect_posterior_density_plot,
)

## Phase 1 — the causal question and the structural model

A target series and two control series share a common latent trend; only the
target receives a **known** effect from day 90 onward (see `src/datasets.py` —
the effect size is fixed so every later phase can be checked against ground
truth). We fit a Bayesian structural time series model — a local-level random
walk plus a regression on the controls — using **only** the pre-intervention
data. The model never sees the post-intervention target; that is what makes its
forward forecast a genuine counterfactual rather than a fitted curve.

In [ ]:
data = load_dataset(n_pre=90, n_post=30, seed=0)
(y_pre, X_pre), (y_post, X_post) = data.split()
print(f"true effect/day: {data.true_effect_per_day}, true cumulative effect: {data.true_cumulative_effect}")
data.as_frame().head()

### Exercise — `src/_01_structural_model.py :: build_local_level_model`

Open `src/_01_structural_model.py`, read the `TODO(human)` block above the
function, implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_structural_model import build_local_level_model

model = build_local_level_model(y_pre, X_pre)
model

## Phase 2 — posterior sampling and the counterfactual forecast

We sample the posterior (a small, fast MCMC run — see `src/_02_counterfactual.py`'s
`run_mcmc`, fully scaffolded), check basic convergence with `arviz`'s trace plot
(a **real** matplotlib Axes — `xy` cannot draw arviz's diagnostic plots, see
`CLAUDE.md` § Plotting), then forecast the local level forward through the
post-period for every posterior draw at once.

In [ ]:
from src._02_counterfactual import run_mcmc

idata = run_mcmc(model)
az.summary(idata, var_names=["sigma_level", "sigma_obs", "beta"])

`arviz.plot_trace` builds its own matplotlib figure/axes internally — this is
the one place in the notebook we import real matplotlib directly, kept separate
from every `xy.pyplot` call (see `CLAUDE.md` § Plotting for why the two never mix).

In [ ]:
import matplotlib.pyplot as mplt

az.plot_trace(idata, var_names=["sigma_level", "sigma_obs", "beta"])
mplt.tight_layout()
mplt.show()

### Exercise — `src/_02_counterfactual.py :: counterfactual_forecast`

Open `src/_02_counterfactual.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_counterfactual import counterfactual_forecast

rng = np.random.default_rng(1)
cf_draws = counterfactual_forecast(idata, X_post, rng)
print(f"counterfactual draws shape: {cf_draws.shape}")
cf_draws.mean(axis=0)[:5]

## Phase 3 — the pointwise and cumulative effect posteriors

Subtracting the (fixed) observed post-period series from the counterfactual
*draws* propagates the full posterior into the effect — no normal-approximation
step. This is the epistemological contrast with practice 105's synthetic
control: there, one set of donor weights gives one counterfactual path and one
effect number; here, the effect is a distribution from the start.

### Exercise — `src/_03_effect.py :: cumulative_effect`

Open `src/_03_effect.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._03_effect import cumulative_effect

effect = cumulative_effect(y_post, cf_draws)
total_effect = effect.cumulative[:, -1]
print(f"true cumulative effect:   {data.true_cumulative_effect:.2f}")
print(f"posterior mean:            {total_effect.mean():.2f}")
print(f"P(total effect > 0):       {(total_effect > 0).mean():.3f}")

## Phase 4 — the highest density interval, and the headline plots

A 94% credible interval is not unique — the HDI is the *narrowest* interval
covering that mass, which matters whenever the effect posterior is skewed.
`xy` has no built-in HDI (or KDE) primitive, so we implement it once and reuse
it for every band/shade below.

### Exercise — `src/_04_hdi.py :: hdi`

Open `src/_04_hdi.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._04_hdi import hdi

cf_lo, cf_hi = zip(*(hdi(cf_draws[:, i]) for i in range(cf_draws.shape[1])))
cum_lo, cum_hi = zip(*(hdi(effect.cumulative[:, i]) for i in range(effect.cumulative.shape[1])))
total_lo, total_hi = hdi(total_effect)
print(f"94% HDI of total effect: ({total_lo:.2f}, {total_hi:.2f})")

In [ ]:
fig = observed_vs_counterfactual_plot(
    data.t, data.y, data.intervention_day,
    cf_draws.mean(axis=0), np.array(cf_lo), np.array(cf_hi),
)
fig

In [ ]:
fig = cumulative_effect_plot(
    data.t[data.intervention_day:], effect.cumulative.mean(axis=0),
    np.array(cum_lo), np.array(cum_hi), data.true_cumulative_effect,
)
fig

## Phase 5 — posterior predictive checks (model criticism)

Before trusting the effect posterior, check that the *model itself* fits the
data it was trained on: draw simulated pre-period series from the fitted model
and compare them against the real pre-period series. Systematic mismatch here
would mean the counterfactual forecast (Phase 2) is extrapolating from a model
that was already a poor description of the past — no amount of posterior
uncertainty fixes a misspecified structural model.

In [ ]:
with model:
    ppc = __import__("pymc").sample_posterior_predictive(idata, progressbar=False, random_seed=0)

az.plot_ppc(ppc, data_pairs={"y_obs": "y_obs"})
mplt.tight_layout()
mplt.show()

## Phase 6 — end-to-end: the posterior density of the total effect

The final object: not a point estimate and a standard error, but a full
posterior over "how much did the intervention change the outcome, in total" —
with the 94% HDI shaded and the (synthetic, known) ground truth marked for
comparison.

In [ ]:
fig = effect_posterior_density_plot(total_effect, total_lo, total_hi, data.true_cumulative_effect)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert cf_draws.shape == (total_effect.shape[0], data.n_post)
assert effect.cumulative.shape == cf_draws.shape
assert total_lo < total_hi
# The true cumulative effect should fall inside (or very near) the 94% HDI —
# the posterior recovering the known ground truth is the pedagogical check.
assert total_lo - 5.0 <= data.true_cumulative_effect <= total_hi + 5.0, (
    "true cumulative effect should be close to the recovered 94% HDI"
)
print("OK")